# Notebook 21 — what *kind* of wrong are the remaining errors?

`genuinely_wrong` is the largest category after `correct_robust` (Qwen 104,
Pixtral 130) and nothing subdivided it, so *"the model is wrong 35–43% of the
time"* carried no explanation. This turns that into counts.

The reviewer question it answers: **"are these real model mistakes, or parser
artifacts?"**

`pilot.failures` proposes a reason per item from text signals. **It is a
pre-sorter, not a verdict** — every label carries the evidence that produced
it, and `needs_visual` is the default rather than a guess:

| label | decided by |
|---|---|
| `extraction_issue` | text — the scoring machinery is at fault |
| `copied_wrong_line` | text — the answer appears verbatim earlier in the page |
| `hallucination` | text — almost no content shared with the truth |
| `notation_misread` | **candidate**, flagged not asserted |
| `needs_visual` | **you**, from the image |

**The counts moved 34% → 23% → 15% for `extraction_issue` as two
false-positive rules were tightened** (short numeric ground truths, then
SymPy's own function names reading as prose). Both are pinned in
`pilot/tests/test_failures.py`. That sensitivity is exactly why the human
pass matters and why the proposal is kept separate from the verdict.

This notebook writes a **coding sheet** and **contact sheets for the audit
sample only**, so the manual pass is ~35 items with a proposed label to
confirm or correct — not 104 read cold.

In [ ]:
# Auth + code access. No GPU/model needed -- this notebook only reads the
# dataset and existing results CSVs, it never runs generation.
import json
import os
import sys

from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
RESULTS_DIR = f"{PROJECT_DIR}/results"

# Reuses the token already cached on Drive by earlier notebooks.
with open(f"{PROJECT_DIR}/.tokens.json") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/
# antlr4 pin: without it SymPy's LaTeX parser fails at CALL time, silently
# degrading every label to the plain-text tier. See
# pilot.canonicalize.latex_parser_available -- this cost 43/300 items once.
%pip install -q "antlr4-python3-runtime==4.11"

sys.path.insert(0, os.path.abspath("repo"))

# Purge any pilot.* left over from a previous clone in this runtime.
# importlib.invalidate_caches() does NOT reload already-imported modules, and
# a stale one produced a KeyError on the 2026-08-08 notebook-13 run for a
# symbol that was demonstrably on disk.
for _name in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_name]
import importlib
importlib.invalidate_caches()

import pilot.canonicalize
import pilot.data
import pilot.parsing
import pilot.plotting
import pilot.rescore

print(f"pilot package imported from: {os.path.dirname(pilot.rescore.__file__)}")
assert pilot.canonicalize.latex_parser_available(), (
    "SymPy's LaTeX parser is NOT working. Every label falls back to plain text, "
    "which inflates entropy and deflates accuracy, and the numbers below will "
    "not match the offline analysis. Fix the antlr4 pin before continuing."
)
print("SymPy LaTeX parser: OK")

: 

In [ ]:
import ast

import pandas as pd

RUNS = {
    "Qwen2.5-VL-3B": "scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv",
    "Pixtral-12B":   "pixtral_perception_full_n300_pixtral-12b_20260809T211028Z.csv",
}
SEED, N_ITEMS, ERROR_FRAC = 42, 300, 0.5

runs = {}
for name, fname in RUNS.items():
    runs[name] = pd.read_csv(f"{RESULTS_DIR}/{fname}")
    print(f"{name:15s} {len(runs[name])} rows, "
          f"model={runs[name]['model_id'].unique().tolist()}, "
          f"K={runs[name]['k_transcription'].unique().tolist()}")

sample = pilot.data.load_fermat_balanced(
    n=N_ITEMS, seed=SEED, target_error_frac=ERROR_FRAC)

# load_fermat_balanced SHUFFLES its final selection, so index alignment is an
# assumption to verify, not one to make. Checking the question text pins the
# row-to-image mapping every display below depends on -- for BOTH runs, since
# the viewer takes images from the sample and text from whichever CSV.
for name, df in runs.items():
    assert len(sample) == len(df), f"{name}: {len(sample)} items vs {len(df)} rows"
    bad = [i for i in range(len(df))
           if sample[i]["orig_q"].strip() != str(df.iloc[i]["orig_q"]).strip()]
    assert not bad, (
        f"{name}: {len(bad)} rows where the rebuilt sample's question does not "
        f"match the CSV's (first: {bad[:5]}). Images would be attached to the "
        "wrong rows -- do not trust anything past this cell until resolved.")
print("\nsample order matches both CSVs on all 300 rows -- images are index-aligned")

In [ ]:
# `classified` and `show_item` come from notebook 17; both are needed by
# the detailed viewer further down. Duplicated deliberately -- every
# notebook in this project is self-contained.
classified, summaries = {}, []
for name, df in runs.items():
    c = pilot.rescore.classify_scoring_outcome(df, progress=True)
    classified[name] = c
    summaries.append(pilot.rescore.scoring_category_summary(c, label=name))
    assert len(c) == len(df) and c["category"].notna().all()

summary = pd.concat(summaries, ignore_index=True)

for name in runs:
    s = summary[summary.model == name]
    print(f"=== {name} ===")
    view = s[["category", "n", "share", "mean_entropy", "frac_multi_tier"]].copy()
    view["share"] = (view["share"] * 100).round(1).astype(str) + "%"
    view["mean_entropy"] = view["mean_entropy"].round(3)
    view["frac_multi_tier"] = (view["frac_multi_tier"] * 100).round(0).astype("Int64").astype(str) + "%"
    print(view.to_string(index=False))
    print(f"  total {int(s.n.sum())}   "
          f"later_regression (fixed then broken again): "
          f"{int(classified[name].later_regression.sum())}\n")

In [ ]:
IMAGE_DIR = f"{PROJECT_DIR}/scoring_inspection_images"
os.makedirs(IMAGE_DIR, exist_ok=True)


def show_item(i, model="Qwen2.5-VL-3B", rules=("strict_v1", "final_term_v4"),
              show_image=True, save=True, raw_chars=320):
    df = runs[model]
    row = df.iloc[i]
    cat = classified[model].loc[i]
    samples_raw = ast.literal_eval(row["all_transcription_samples_raw"])

    print("#" * 100)
    print(f"# ITEM {i}   [{model}]   category = {cat['category']}"
          + (f"   attributed to: {cat['attributed_bug']}"
             if pd.notna(cat["attributed_bug"]) else ""))
    print(f"#   has_error={bool(row['has_error'])}   "
          f"handwriting_style={row['handwriting_style']}   "
          f"image_quality={row['image_quality']}   "
          f"extractor branches used: {cat['n_distinct_tiers']}"
          + ("   later_regression=True" if cat["later_regression"] else ""))
    print("#   verdict by rule: " + "   ".join(
        f"{r}={bool(classified[model].loc[i, 'correct_' + r])}"
        for r in pilot.rescore.RULES))
    print("#" * 100)

    if show_image:
        img = sample[i]["image"]
        if save:
            img.save(f"{IMAGE_DIR}/item{i:03d}.png")
        plt.figure(figsize=(9, 9 * img.height / max(img.width, 1)))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"item {i} — the handwritten page the model was shown")
        plt.show()

    for rule in rules:
        tr = pilot.rescore.trace_item(samples_raw, row["pert_a"], rule)
        print(pilot.rescore.format_trace(
            tr, question=row["orig_q"], category=cat["category"],
            raw_chars=raw_chars))


def first_in(category, model="Qwen2.5-VL-3B", n=1):
    return classified[model].index[
        classified[model]["category"] == category].tolist()[:n]


print("show_item(i, model=...) ready.")
for name in runs:
    print(f"  {name}: " + "  ".join(
        f"{c}={len(classified[name][classified[name].category == c])}"
        for c in pilot.rescore.CATEGORIES))

In [ ]:
# Pre-sort every genuinely_wrong item on both models. ~4 min (SymPy).
import pilot.failures

presorted, counts = {}, []
for name, df in runs.items():
    ps = pilot.failures.presort(df, progress=True)
    presorted[name] = ps
    row = {"model": name, "n_genuinely_wrong": len(ps)}
    for lab in pilot.failures.LABELS:
        row[lab] = int((ps["label"] == lab).sum())
    counts.append(row)

counts = pd.DataFrame(counts)
print(counts.to_string(index=False))
print()
for name, ps in presorted.items():
    share = (ps["label"].value_counts(normalize=True)
             .reindex(pilot.failures.LABELS).fillna(0))
    print(f"{name:15s} " + "  ".join(f"{k}={v:.0%}" for k, v in share.items()))

### Reading the counts

- **`hallucination` is 0 on both models.** The model never invents content
  unrelated to the page — worth stating, because it is the failure a reviewer
  most fears from a VLM asked to transcribe.
- **`extraction_issue` ≈15% on both** (Qwen 15.4%, Pixtral 14.6%). So even
  after four scoring rules, roughly one in seven "model errors" is still the
  scoring machinery. That number is a *lower bound* — it only counts what is
  detectable from text.
- **`needs_visual` ≈57% on both.** That is the honest remainder and what the
  rest of this notebook is for.

In [ ]:
# The audit sample: ~35 items, selected BY RULE across the strata that
# matter, so it is reproducible and cannot be tuned. High- and low-entropy
# wrong items both appear -- a confidently wrong page and a maximally
# uncertain one fail for different reasons.
MODEL = "Qwen2.5-VL-3B"

audit = pilot.failures.select_audit_sample(presorted[MODEL], n_per_stratum=8)
print(f"{len(audit)} items across {audit['stratum'].nunique()} strata\n")
print(audit.groupby("stratum", observed=True)["label"]
      .value_counts().unstack(fill_value=0).to_string())

AUDIT_DIR = f"{PROJECT_DIR}/figures/failure_audit"
os.makedirs(AUDIT_DIR, exist_ok=True)
sheet = pilot.failures.coding_sheet(audit, f"{AUDIT_DIR}/coding_sheet.csv")
print(f"\ncoding sheet -> {AUDIT_DIR}/coding_sheet.csv")
print("Fill in final_label (and notes) while reading the sheets below. "
      "proposed_label is left untouched on purpose -- keeping both is how we "
      "measure how often the pre-sorter was wrong.")

In [ ]:
# Contact sheets for the AUDIT SAMPLE ONLY, captioned with the proposed
# label so the page and the proposal are visible together.
#
# Written as files, never shown inline: on the 2026-08-10 notebook-17 run
# every plt.show() produced nothing in the synced copy (zero image/png, no
# error, no repo-side cause). A saved PNG is immune.
import matplotlib.pyplot as plt

def caption(r):
    return (f"item {int(r['item'])}   H={r['entropy']:.2f}   [{r['stratum']}]\n"
            f"PROPOSED: {r['label']}\n"
            f"model: {str(r['model_label'])[:40]}\n"
            f"truth: {str(r['gt_label'])[:40]}")

figs = pilot.plotting.contact_sheet(
    [sample[int(r["item"])]["image"] for _, r in audit.iterrows()],
    [caption(r) for _, r in audit.iterrows()],
    ncols=3, per_page=9,
    title=f"Failure audit sample -- {MODEL} (proposed labels, confirm or correct)")
for page, fig in enumerate(figs, 1):
    path = f"{AUDIT_DIR}/audit_p{page}.png"
    fig.savefig(path, dpi=150, facecolor=fig.get_facecolor())
    plt.close(fig)
print(f"{len(audit)} items -> {len(figs)} page(s) in {AUDIT_DIR}")
print("open: My Drive > uncertainty-math-vlm > figures > failure_audit")

In [ ]:
# Full detail for any single item, when the contact sheet is not enough:
# the image, all five raw outputs, what the parser took from each, what it
# took from the ground truth, and the column where the two labels diverge.
for i in audit[audit.stratum == "needs_visual"]["item"].head(2):
    show_item(int(i), model=MODEL)

### Filling in the sheet

For each item, decide which it is and write it in `final_label`:

| if the page shows... | label |
|---|---|
| the writing genuinely cannot be read | `bad_handwriting` |
| the model read a symbol wrong (×/x, +/−, exponent) | `notation_misread` |
| the model transcribed a real line, but the wrong one | `copied_wrong_line` |
| the model's answer is right and the *truth label* is broken | `extraction_issue` |
| the model wrote something not on the page at all | `hallucination` |

**Two things to watch, both already observed on this data:**

- **The model sometimes silently corrects the injected error.** Item 55's
  ground truth is `1 + \tan x \tan y` (FERMAT's injected sign flip) and both
  models wrote the textbook-correct `1 - \tan x \tan y`. That is a *real*
  perception failure, not a misread — label it `notation_misread` and note it.
- **A page can be scored wrong while the model is right.** Item 218 is the
  clearest: both models answered `option a` correctly and the ground-truth
  extractor mangled `Option $\text{A}$` into `sympy:a`. That is
  `extraction_issue`.

In [ ]:
# Once final_label is filled in, re-read the sheet and produce the counts.
# Run this AFTER the manual pass.
final = pd.read_csv(f"{AUDIT_DIR}/coding_sheet.csv")
# fillna FIRST: pandas reads the empty column back as NaN, and str(NaN) is
# the string "nan", so a naive .astype(str) counts every uncoded row as coded.
final["final_label"] = final["final_label"].fillna("").astype(str).str.strip()
final["proposed_label"] = final["proposed_label"].fillna("").astype(str).str.strip()
done = final[final["final_label"] != ""]

if done.empty:
    print("coding sheet not filled in yet -- nothing to count.")
else:
    print(f"coded {len(done)}/{len(final)} items\n")
    print("FINAL FAILURE CATEGORIES")
    print(done["final_label"].value_counts().to_string())

    # How often was the pre-sorter right? Worth reporting alongside the
    # counts -- it is the difference between "we categorised these" and "a
    # regex categorised these".
    agree = (done["final_label"] == done["proposed_label"]).mean()
    print(f"\npre-sorter agreed with the human label on {agree:.0%} of coded items")
    print(pd.crosstab(done["proposed_label"], done["final_label"]).to_string())

## What to carry out

- **Report the counts from `final_label`, not `proposed_label`.** The
  pre-sorter's own accuracy goes next to them; that is the difference between
  "we categorised these" and "a regex categorised these".
- **`hallucination = 0` is a result in itself** and should be stated.
- **`extraction_issue` is a lower bound** — text signals only see what text
  can see, and the visual pass will likely find more.
- If the coded `extraction_issue` share is much above 15%, the headline
  accuracy is understated by more than the sensitivity table suggests, and
  that belongs in Limitations.